# Prerequisite

In [1]:
# GPU 런타임 확인
!nvidia-smi

Tue Sep  1 07:38:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
# nsight 도구 설치
# nsight 도구는 일반적으로 NVIDIA CUDA Toolkit의 일부이므로, nvidia-cuda-toolkit 패키지를 설치
!apt-get -qq update && apt-get -qq install -y nvidia-cuda-toolkit

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Preconfiguring packages ...
Selecting previously unselected package libdebuginfod-common.
(Reading database ... 122579 files and directories currently installed.)
Preparing to unpack .../00-libdebuginfod-common_0.186-1ubuntu0.1_all.deb ...
Unpacking libdebuginfod-common (0.186-1ubuntu0.1) ...
Selecting previously unselected package libatspi2.0-0:amd64.
Preparing to unpack .../01-libatspi2.0-0_2.44.0-3_amd64.deb ...
Unpacking libatspi2.0-0:amd64 (2.44.0-3) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../02-libxtst6_2%3a1.2.3-1build4_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1build4) ...
Selecting previously unselected package session-migration.
Preparing to unpack .../03-session-migration_0.3.6_amd64.deb ...
Unpacking s

# nsys

In [12]:
# nsys 확인 및 경로 설정
import os
nsys_path = "/opt/nvidia/nsight-compute/2025.1.1/host/target-linux-x64"
os.environ["PATH"] += os.pathsep + nsys_path
print(f"nsys path added to PATH: {nsys_path}")
!which nsys
!nsys --version

nsys path added to PATH: /opt/nvidia/nsight-compute/2025.1.1/host/target-linux-x64
/opt/nvidia/nsight-compute/2025.1.1/host/target-linux-x64/nsys
NVIDIA Nsight Systems version 2025.1.1.0


In [13]:
# 2. 아주 작은 워크로드
%%writefile tiny.py
import torch
x = torch.randn(4096, 4096, device="cuda")
y = torch.randn(4096, 4096, device="cuda")
for _ in range(20):
    z = x @ y
    z = torch.relu(z)
torch.cuda.synchronize()
print(z.sum().item())

Overwriting tiny.py


In [14]:
# 3. 감싸서 실행
!nsys profile -t cuda,nvtx,osrt -o /content/demo --force-overwrite=true python tiny.py

Invalid plugin configuration: Executable path does not exist: /opt/nvidia/nsight-compute/2025.1.1/host/target-linux-x64/plugins/efa_metrics/nic_sampler
428277568.0
Generating '/tmp/nsys-report-31a2.qdstrm'
[1/1] [========================100%] demo.nsys-rep
Generated:
    /content/demo.nsys-rep


In [19]:
# 4. 내려받기
from google.colab import files
files.download("/content/demo.nsys-rep")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# ncu

In [20]:
# ncu 버전 확인
!which ncu && ncu --version

/usr/local/cuda/bin/ncu
NVIDIA (R) Nsight Compute Command Line Profiler
Copyright (c) 2018-2025 NVIDIA Corporation
Version 2025.1.1.0 (build 35528883) (public-release)


In [21]:
# GPU 지원 여부 확인
!ncu --list-chips | head

ad102, ad103, ad104, ad106, ad107, ga100, ga102, ga103, ga104, ga106, ga107, ga10b, gb100, gb10b, gb202, gb203, gb205, gh100, gv100, gv11b, tu102, tu104, tu106, tu116, tu117


In [22]:
# ncu 감싸서 실행
!ncu -c 5 --set basic -o /content/demo_ncu -f python tiny.py

==PROF== Connected to process 10124 (/usr/bin/python3.13)
==PROF== Profiling "distribution_elementwise_grid..." - 0 (1/5): 0%....50%....100% - 9 passes
==PROF== Profiling "distribution_elementwise_grid..." - 1 (2/5): 0%....50%....100% - 9 passes
==PROF== Profiling "volta_sgemm_128x64_nn" - 2 (3/5): 0%....50%....100% - 9 passes
==PROF== Profiling "vectorized_elementwise_kernel" - 3 (4/5): 0%....50%....100% - 9 passes
==PROF== Profiling "volta_sgemm_128x64_nn" - 4 (5/5): 0%....50%....100% - 9 passes
428079552.0
==PROF== Disconnected from process 10124
==PROF== Report: /content/demo_ncu.ncu-rep


In [23]:
# 결과 다운로드
from google.colab import files
files.download("/content/demo_ncu.ncu-rep")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>